In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re


def parse_stat(text):
    if pd.isna(text): return 0.0
    if isinstance(text, (int, float)): return float(text)
    try:
        return float(re.split('±', str(text))[0])
    except:
        return 0.0


df = pd.read_csv('MIL/all_metrics_summary.csv')

target_method = 'SPE'
methods_list = ['AB_MIL', 'TRANS_MIL', 'CLAM_MB_MIL', 'CLAM_SB_MIL', 'WIKG_MIL',
                'MAMBA2D_MIL', 'AEM_MIL', 'MICO_MIL', 'MSM_MIL', 'TDA_MIL', 'GDF_MIL', 'SPE']
df = df[df['method'].isin(methods_list)].reset_index(drop=True)

perf_metrics = ['acc', 'bacc', 'macro_auc', 'macro_f1', 'macro_recall', 'quadratic_kappa']
metrics_map = {
    'acc': 'Accuracy', 'bacc': 'Bal. Acc.', 'macro_auc': 'Macro AUC',
    'macro_f1': 'Macro F1', 'macro_recall': 'Macro Recall', 'quadratic_kappa': 'Quad. Kappa'
}

colors = {
    'top1': '#D32F2F', 'top2': '#2E7D32', 'top3': '#1565C0',
    'target_out': '#5E35B1',  # 跌出前三时高亮深紫色
    'others': '#E0E4E8', 'range': '#F4F5F7'
}

fig, ax = plt.subplots(figsize=(14, 8.5), dpi=120)
sns.set_style("white")

for i, m in enumerate(perf_metrics):
    temp_df = df[['method', m]].copy()
    temp_df['val'] = temp_df[m].apply(parse_stat)

    # 严格绝对排序
    sorted_df = temp_df.sort_values('val', ascending=False).reset_index(drop=True)

    # 归一化 X 坐标映射
    v_min, v_max = sorted_df['val'].min(), sorted_df['val'].max()
    denom = (v_max - v_min) if (v_max - v_min) > 1e-9 else 1e-9
    sorted_df['plot_x'] = 0.18 + (sorted_df['val'] - v_min) / denom * 0.67

    ax.hlines(y=i, xmin=0.15, xmax=0.88, color=colors['range'], linewidth=16, zorder=1, capstyle='round')

    top3_df = sorted_df.iloc[:3]
    others_df = sorted_df.iloc[3:]

    target_in_top3 = target_method in top3_df['method'].values

    if not target_in_top3:
        target_row = others_df[others_df['method'] == target_method]
        others_df = others_df[others_df['method'] != target_method]

    ax.scatter(others_df['plot_x'], [i] * len(others_df), color=colors['others'], s=55, alpha=0.7, zorder=2)

    plot_queue = []
    for rank, (_, row) in enumerate(top3_df.iterrows()):
        plot_queue.append((f'top{rank + 1}', row))

    if not target_in_top3 and not target_row.empty:
        plot_queue.append(('target_out', target_row.iloc[0]))

    occupied_x = []

    for style_key, row in plot_queue:
        px = row['plot_x']
        val = row['val']
        name = row['method'].replace('_MIL', '')
        is_target = (row['method'] == target_method)

        ax.scatter(px, i, color=colors[style_key], s=110, edgecolors='white', linewidths=1.5, zorder=4)

        ha_style = 'center'
        text_x = px

        # 遍历检查是否与已有的文字标签 X 轴冲突
        for past_x in occupied_x:
            if abs(px - past_x) < 0.03:
                # 如果当前点在已有点的左侧，文字继续往左挪并右对齐；反之亦然
                if px < past_x:
                    text_x = px - 0.018
                    ha_style = 'right'
                else:
                    text_x = px + 0.025
                    ha_style = 'left'
                past_x = text_x
                break

        occupied_x.append(text_x)

        font_w = 'bold' if is_target else 'semibold'
        ax.text(text_x, i - 0.28, f"{name}\n{val:.4f}", color=colors[style_key],
                ha=ha_style, va='top', fontsize=9, fontweight=font_w,
                bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none', alpha=0.85))

ax.set_yticks(range(len(perf_metrics)))
ax.set_yticklabels([metrics_map[m] for m in perf_metrics], fontsize=12.5, fontweight='bold', color='#222222')
ax.set_xticks([])
ax.set_xticklabels([])
ax.tick_params(left=False)
sns.despine(left=True, bottom=True)

ax.set_ylim(-0.5, len(perf_metrics) - 0.2)
ax.set_xlim(0.13, 0.95)

from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], marker='o', color='white', markerfacecolor=colors['top1'], markersize=10, label='Top-1'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor=colors['top2'], markersize=10, label='Top-2'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor=colors['top3'], markersize=10, label='Top-3'),
    # Line2D([0], [0], marker='o', color='white', markerfacecolor=colors['target_out'], markersize=10, label=f'{target_method} (Out of Top-3)'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor=colors['others'], markersize=8, label='Other Baselines')
]

leg = ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.015),
                ncol=5, frameon=True, fontsize=10.5, edgecolor='#F0F0F0', facecolor='white')
leg.get_frame().set_linewidth(1.0)
for text in leg.get_texts():
    text.set_color("#333333")
    text.set_weight("semibold")

plt.suptitle('MIL Method Comparison', fontsize=15, fontweight='bold', x=0.5, y=0.92)
plt.savefig('./result-int/mil-contrast.svg', dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches


types = ['CNB', 'RP', 'TURP', 'Total']
metrics = ['acc', 'bacc', 'quadratic_kappa', 'macro_auc', 'macro_f1', 'macro_recall']
metric_labels = ['Accuracy', 'Balance\nAcc.', 'Quadratic\nKappa', 'Macro\nAUC', 'Macro\nF1', 'Macro\nRecall']


df = pd.read_csv('MIL/Ensemble/Seed/final_evaluation_summary1.csv')
data = df[['acc','bacc','quadratic_kappa','macro_auc','macro_f1','macro_recall']].values
COLOR_CNB = '#2E8B57'    # 海绿色 - 完美分类
COLOR_RP = '#4682B4'     # 钢蓝色
COLOR_TURP = '#CD853F'   # 秘鲁色 - 挑战性
COLOR_TOTAL = '#6A5ACD'  # 石板蓝

colors = [COLOR_CNB, COLOR_RP, COLOR_TURP, COLOR_TOTAL]

fig, ax = plt.subplots(figsize=(16, 9))

n_metrics = len(metrics)
n_types = len(types)
x = np.arange(n_metrics)
width = 0.18  # 柱宽

# 绘制分组柱状图
for i, (type_name, color) in enumerate(zip(types, colors)):
    offset = (i - n_types/2 + 0.5) * width
    bars = ax.bar(x + offset, data[i], width,
                  label=type_name, color=color,
                  edgecolor='white', linewidth=0.8,
                  alpha=0.9, zorder=3)

    for j, (bar, val) in enumerate(zip(bars, data[i])):
        if val == 1.0:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                   '1.00', ha='center', va='bottom', fontsize=8.5,
                   fontweight='bold', color=color)
        else:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                   f'{val:.3f}' if val < 0.95 else f'{val:.2f}',
                   ha='center', va='bottom', fontsize=7.5, color='#333333')

for j in range(n_metrics):
    offset = (0 - n_types/2 + 0.5) * width
    x_pos = x[j] + offset
    ax.annotate('★', xy=(x_pos, 1.0), xytext=(x_pos, 1.06), fontsize=14, color=COLOR_CNB, ha='center',
                fontweight='bold')


turp_recall_idx = 5
turp_recall_val = data[2, turp_recall_idx]
offset_turp = (2 - n_types/2 + 0.5) * width
x_turp = x[turp_recall_idx] + offset_turp


ax.axhline(y=0.9, color='#CCCCCC', linestyle='--', linewidth=1, alpha=0.7, zorder=1)
ax.text(n_metrics - 0.5, 0.905, '0.9', fontsize=9, color='#999999', ha='right')

ax.axhline(y=0.95, color='#CCCCCC', linestyle='--', linewidth=1, alpha=0.7, zorder=1)
ax.text(n_metrics - 0.5, 0.955, '0.95', fontsize=9, color='#999999', ha='right')

ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('Cross-Specimen Generalization: CNB, RP, and TURP', fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=10.5)
ax.set_ylim(0.65, 1.18)
ax.set_xlim(-0.5, n_metrics - 0.5)

# 图例
legend = ax.legend(loc='upper left', fontsize=11, frameon=True, fancybox=True, shadow=True, ncol=2)
legend.get_frame().set_facecolor('#FAFAFA')
legend.get_frame().set_edgecolor('#CCCCCC')

# 网格
ax.grid(axis='y', alpha=0.3, linestyle='--', zorder=0)
ax.set_axisbelow(True)

# 背景
ax.set_facecolor('#FAFAFA')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')


plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('result-int/specimen_perf.svg', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import matplotlib.gridspec as gridspec
import os

# ============================================================
# 1. Simulate realistic data matching the provided statistics
# ============================================================
np.random.seed(42)

spe_path = 'bootstrap/SPE/real_inference_bootstrap_100_accuracy_all.csv'
spe_all =pd.read_csv(spe_path)['accuracy'].values
spe_turp_path = 'bootstrap/SPE/real_inference_bootstrap_100_accuracy_TURP.csv'
spe_turp =pd.read_csv(spe_turp_path)['accuracy'].values
ste_path = 'bootstrap/ste/real_inference_bootstrap_100_accuracy_all.csv'
ste_all =pd.read_csv(ste_path)['accuracy'].values
ste_turp_path = 'bootstrap/ste/real_inference_bootstrap_100_accuracy_TURP.csv'
ste_turp =pd.read_csv(ste_turp_path)['accuracy'].values


# Calculate statistics
ste_ci = np.percentile(ste_all, [2.5, 97.5])
spe_ci = np.percentile(spe_all, [2.5, 97.5])
ste_cv = np.std(ste_all) / np.mean(ste_all)
spe_cv = np.std(spe_all) / np.mean(spe_all)
cv_reduction = (1 - spe_cv / ste_cv) * 100

print(f"Standard Ensemble: Mean={np.mean(ste_all):.4f}, Std={np.std(ste_all):.4f}, CV={ste_cv:.4f}")
print(f"SPE: Mean={np.mean(spe_all):.4f}, Std={np.std(spe_all):.4f}, CV={spe_cv:.4f}")
print(f"CV Reduction: {cv_reduction:.1f}%")

# ============================================================
# 2. Create figure
# ============================================================
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, 
                       height_ratios=[3, 1], 
                       width_ratios=[4, 1],
                       hspace=0.35, wspace=0.25)

ax_main = fig.add_subplot(gs[:, 0])
ax_density = fig.add_subplot(gs[0, 1])
ax_inset = fig.add_subplot(gs[1, 1])

COLOR_ENSEMBLE = '#8B8B8B'
COLOR_SPE = '#C41E3A'
COLOR_BG = '#FAFAFA'

# ============================================================
# 3. Main plot: Violin + KDE density
# ============================================================
data_list = [ste_all, spe_all]
labels = ['Standard Ensemble', 'SPE']
colors = [COLOR_ENSEMBLE, COLOR_SPE]
positions = [1, 2]

# Violin plot
violin_parts = ax_main.violinplot(data_list, positions=positions, 
                                   widths=0.6, showmeans=False, 
                                   showmedians=False, showextrema=False)

for i, pc in enumerate(violin_parts['bodies']):
    pc.set_facecolor(colors[i])
    pc.set_edgecolor(colors[i])
    pc.set_alpha(0.25)
    pc.set_linewidth(2)

# KDE density overlay
x_min = min(ste_all.min(), spe_all.min()) - 0.005
x_max = max(ste_all.max(), spe_all.max()) + 0.005
x_range = np.linspace(x_min, x_max, 500)

kde_ste = stats.gaussian_kde(ste_all)
kde_ste.set_bandwidth(bw_method=0.25)
density_ste = kde_ste(x_range)
density_ste_norm = density_ste / np.max(density_ste) * 0.35

kde_spe = stats.gaussian_kde(spe_all)
kde_spe.set_bandwidth(bw_method=0.25)
density_spe = kde_spe(x_range)
density_spe_norm = density_spe / np.max(density_spe) * 0.35

ax_main.fill_betweenx(x_range, 1 - density_ste_norm, 1 + density_ste_norm, 
                       color=COLOR_ENSEMBLE, alpha=0.6, zorder=3)
ax_main.fill_betweenx(x_range, 2 - density_spe_norm, 2 + density_spe_norm, 
                       color=COLOR_SPE, alpha=0.6, zorder=3)

ax_main.plot(1 - density_ste_norm, x_range, color=COLOR_ENSEMBLE, linewidth=2, zorder=4)
ax_main.plot(1 + density_ste_norm, x_range, color=COLOR_ENSEMBLE, linewidth=2, zorder=4)
ax_main.plot(2 - density_spe_norm, x_range, color=COLOR_SPE, linewidth=2, zorder=4)
ax_main.plot(2 + density_spe_norm, x_range, color=COLOR_SPE, linewidth=2, zorder=4)

# Boxplot elements
for i, data in enumerate(data_list):
    pos = positions[i]
    q1, median, q3 = np.percentile(data, [25, 50, 75])
    
    ax_main.plot([pos - 0.08, pos + 0.08], [median, median], 
                 color='white', linewidth=3, zorder=5)
    ax_main.plot([pos - 0.08, pos + 0.08], [median, median], 
                 color='black', linewidth=1.5, zorder=6)
    ax_main.plot([pos, pos], [q1, q3], color='black', linewidth=4, zorder=5, alpha=0.8)
    
    mean_val = np.mean(data)
    ax_main.scatter(pos, mean_val, color='white', s=80, zorder=7, 
                   edgecolor='black', linewidth=1.5)

# 95% CI annotations
ax_main.hlines(y=ste_ci, xmin=0.65, xmax=1.35, colors=COLOR_ENSEMBLE, 
               linestyles='--', linewidths=1.5, alpha=0.7)
ax_main.annotate(f'95% CI: [{ste_ci[0]:.4f}, {ste_ci[1]:.4f}]', 
                 xy=(1, ste_ci[1]), xytext=(0.4, ste_ci[1] + 0.005),
                 fontsize=9, color=COLOR_ENSEMBLE, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=COLOR_ENSEMBLE, lw=1))

ax_main.hlines(y=spe_ci, xmin=1.65, xmax=2.35, colors=COLOR_SPE, 
               linestyles='--', linewidths=1.5, alpha=0.7)
ax_main.annotate(f'95% CI: [{spe_ci[0]:.4f}, {spe_ci[1]:.4f}]', 
                 xy=(2, spe_ci[1]), xytext=(2.1, spe_ci[1] + 0.005),
                 fontsize=9, color=COLOR_SPE, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=COLOR_SPE, lw=1))

# CV reduction label
box_text = f'CV Reduction\n{cv_reduction:.1f}%'
props = dict(boxstyle='round,pad=0.5', facecolor='#FFF3F3', 
             edgecolor=COLOR_SPE, linewidth=2)
y_pos_label = min(ste_all.min(), spe_all.min()) - 0.015
ax_main.text(1.5, y_pos_label, box_text, fontsize=12, fontweight='bold',
             color=COLOR_SPE, ha='center', va='center', bbox=props)

# Stability arrow
arrow_y = min(ste_all.min(), spe_all.min()) + 0.005
ax_main.annotate('', xy=(2, arrow_y + 0.005), xytext=(1, arrow_y + 0.005),
                arrowprops=dict(arrowstyle='->', color=COLOR_SPE, lw=2.5))
# ax_main.text(1.5, arrow_y + 0.012, 'Stability Improvement', fontsize=10, ha='center', 
#              color=COLOR_SPE, fontweight='bold')

# Main plot settings
ax_main.set_xlim(0.3, 2.7)
ax_main.set_ylim(x_min - 0.03, x_max + 0.02)
ax_main.set_xticks(positions)
ax_main.set_xticklabels(labels, fontsize=12, fontweight='bold')
ax_main.set_ylabel('Accuracy', fontsize=13, fontweight='bold')
ax_main.set_title('Bootstrap Accuracy Distribution (n=100)', 
                  fontsize=15, fontweight='bold', pad=15)
ax_main.grid(axis='y', alpha=0.3, linestyle='--')
ax_main.set_facecolor(COLOR_BG)

# Jittered points
for i, data in enumerate(data_list):
    pos = positions[i]
    jitter = np.random.normal(0, 0.04, len(data))
    ax_main.scatter([pos + j for j in jitter], data, alpha=0.15, s=15, 
                   color=colors[i], zorder=2)

# ============================================================
# 4. Top-right: Density curve comparison
# ============================================================
y_range = np.linspace(x_min, x_max, 500)

kde_ste2 = stats.gaussian_kde(ste_all)
kde_ste2.set_bandwidth(bw_method=0.3)
dens_ste = kde_ste2(y_range)

kde_spe2 = stats.gaussian_kde(spe_all)
kde_spe2.set_bandwidth(bw_method=0.3)
dens_spe = kde_spe2(y_range)

max_dens = max(np.max(dens_ste), np.max(dens_spe))
dens_ste_norm = dens_ste / max_dens * 0.9
dens_spe_norm = dens_spe / max_dens * 0.9

ax_density.fill_betweenx(y_range, 0, dens_ste_norm, 
                         color=COLOR_ENSEMBLE, alpha=0.5, 
                         label='Standard Ensemble')
ax_density.fill_betweenx(y_range, 0, dens_spe_norm, 
                         color=COLOR_SPE, alpha=0.7, 
                         label='SPE')

ax_density.plot(dens_ste_norm, y_range, color=COLOR_ENSEMBLE, linewidth=2)
ax_density.plot(dens_spe_norm, y_range, color=COLOR_SPE, linewidth=2)

peak_ste = y_range[np.argmax(dens_ste)]
peak_spe = y_range[np.argmax(dens_spe)]
ax_density.annotate(f'mean={np.mean(ste_all):.4f}', 
                   xy=(np.max(dens_ste_norm), peak_ste), 
                   xytext=(np.max(dens_ste_norm)+0.15, peak_ste),
                   fontsize=9, color=COLOR_ENSEMBLE, fontweight='bold')
ax_density.annotate(f'mean={np.mean(spe_all):.4f}', 
                   xy=(np.max(dens_spe_norm), peak_spe), 
                   xytext=(np.max(dens_spe_norm)+0.15, peak_spe),
                   fontsize=9, color=COLOR_SPE, fontweight='bold')

ax_density.set_ylim(x_min, x_max)
ax_density.set_xlim(0, 1.3)
ax_density.set_xlabel('Density', fontsize=11)
ax_density.set_title('Density Comparison', fontsize=12, fontweight='bold')
ax_density.set_yticklabels([])
ax_density.grid(axis='y', alpha=0.3)
ax_density.set_facecolor(COLOR_BG)
ax_density.legend(loc='upper left', fontsize=9)

# ============================================================
# 5. Bottom-right: TURP subset inset
# ============================================================
ax_inset.set_title('TURP Subset (Challenging)', fontsize=11, 
                   fontweight='bold', pad=10)

turp_data_list = [ste_turp, spe_turp]
turp_positions = [1, 2]

turp_violin = ax_inset.violinplot(turp_data_list, positions=turp_positions, 
                                   widths=0.5, showmeans=False, 
                                   showmedians=False, showextrema=False)

for i, pc in enumerate(turp_violin['bodies']):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.3)
    pc.set_edgecolor(colors[i])
    pc.set_linewidth(1.5)

for i, data in enumerate(turp_data_list):
    pos = turp_positions[i]
    q1, median, q3 = np.percentile(data, [25, 50, 75])
    ax_inset.plot([pos - 0.06, pos + 0.06], [median, median], 
                 color='black', linewidth=2, zorder=5)
    ax_inset.plot([pos, pos], [q1, q3], color='black', 
                 linewidth=3, zorder=4, alpha=0.7)

turp_ste_ci = np.percentile(ste_turp, [2.5, 97.5])
turp_spe_ci = np.percentile(spe_turp, [2.5, 97.5])

ax_inset.hlines(y=turp_ste_ci, xmin=0.7, xmax=1.3, 
               colors=COLOR_ENSEMBLE, linestyles='--', 
               linewidths=1, alpha=0.6)
ax_inset.hlines(y=turp_spe_ci, xmin=1.7, xmax=2.3, 
               colors=COLOR_SPE, linestyles='--', 
               linewidths=1, alpha=0.6)

turp_min = min(ste_turp.min(), spe_turp.min())
turp_max = max(ste_turp.max(), spe_turp.max())
ax_inset.set_xlim(0.3, 2.7)
ax_inset.set_ylim(turp_min - 0.02, turp_max + 0.02)
ax_inset.set_xticks(turp_positions)
ax_inset.set_xticklabels(['Ensemble', 'SPE'], fontsize=9)
ax_inset.set_ylabel('Accuracy', fontsize=10)
ax_inset.grid(axis='y', alpha=0.3)
ax_inset.set_facecolor(COLOR_BG)

for i, data in enumerate(turp_data_list):
    pos = turp_positions[i]
    jitter = np.random.normal(0, 0.03, len(data))
    ax_inset.scatter([pos + j for j in jitter], data, 
                    alpha=0.12, s=10, color=colors[i], zorder=2)

# ============================================================
# 6. Figure caption (English only)
# ============================================================
# fig.text(0.5, 0.02, 
#          'Figure B: Stability evidence. Based on 100 Bootstrap resampling iterations, SPE demonstrates significantly narrower\n'
#          'Accuracy distribution (CV reduced by ~70%), indicating superior prediction stability compared to standard Ensemble.\n'
#          'Inset shows TURP subset performance under the most challenging scenario.',
#          ha='center', fontsize=10, style='italic', color='#555555',
#          bbox=dict(boxstyle='round,pad=0.5', facecolor='#F5F5F5', 
#                   edgecolor='#CCCCCC'))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('result-int/bootstrap.svg', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Figure B saved successfully")
